# Menyiapkan Dataset Modeling PM2.5 Jakarta

Notebook ini digunakan untuk menyiapkan dataset modeling dari data master kualitas udara Jakarta yang sebelumnya sudah digabung dengan data cuaca.

Input utama notebook ini adalah:

```text
dataset_master_spku_weather.csv
```

Notebook ini berperan sebagai tahap **preprocessing modeling data**, yaitu tahap pengolahan data sebelum masuk ke training model. Fokus utamanya adalah:

1. Membersihkan kolom yang tidak dibutuhkan.
2. Mengubah tipe data agar sesuai.
3. Menangani missing value pada PM2.5, polutan pendukung, dan weather.
4. Membuat fitur waktu, fitur cuaca, fitur lag, dan fitur rolling.
5. Membentuk dataset prediksi untuk beberapa horizon.
6. Menyimpan dataset akhir yang siap digunakan pada notebook modeling.

Output utama notebook ini adalah tiga dataset:

| Dataset | Target Prediksi | Kegunaan |
|---|---|---|
| `dataset_h6.csv` | PM2.5 6 jam ke depan | Modeling horizon pendek |
| `dataset_h12.csv` | PM2.5 12 jam ke depan | Modeling horizon menengah |
| `dataset_h24.csv` | PM2.5 24 jam ke depan | Modeling horizon harian |

Dengan demikian, notebook ini menjadi jembatan antara dataset master hasil scraping/EDA dan dataset final yang siap dipakai untuk training model seperti LightGBM, XGBoost, CatBoost, atau model ensemble lainnya.

## Strategi Cleaning dan Imputasi Data

Tahap awal notebook ini berfokus pada pembersihan data dan penanganan missing value.

Data kualitas udara hasil scraping masih memiliki missing value yang cukup besar, terutama pada kolom `pm25`. Karena target utama proyek ini adalah forecasting PM2.5, maka notebook ini membuat dua versi PM2.5:

| Kolom | Makna |
|---|---|
| `pm25_raw` | Nilai PM2.5 asli sebelum imputasi |
| `pm25` | Nilai PM2.5 final setelah proses cleaning dan imputasi |

Strategi ini penting karena nilai asli tetap disimpan untuk keperluan audit, sedangkan kolom `pm25` digunakan sebagai versi bersih untuk membangun fitur historis dan target horizon.

Secara umum, strategi imputasi yang digunakan adalah:

| Kelompok Variabel | Strategi Imputasi |
|---|---|
| Weather | Interpolasi linear, lalu backward fill dan forward fill |
| Polutan pendukung | Forward fill terbatas, interpolasi linear, backward fill, forward fill, lalu median |
| PM2.5 | Forward fill terbatas, interpolasi linear, backward fill, forward fill, lalu median |

Catatan metodologis penting: karena kolom `pm25` setelah imputasi digunakan untuk membuat target masa depan, maka target pada dataset akhir bukan sepenuhnya berasal dari observasi asli. Sebagian target dapat berasal dari nilai hasil imputasi. Hal ini perlu dijelaskan dalam laporan agar proses modeling tetap transparan.

## Dari Data Bersih ke Fitur Time Series

Setelah data dibersihkan dan missing value ditangani, notebook ini membangun fitur-fitur yang relevan untuk forecasting PM2.5.

Fitur yang dibuat dapat dikelompokkan sebagai berikut:

| Kelompok Fitur | Contoh Fitur | Fungsi |
|---|---|---|
| Fitur waktu | `hour_num`, `dayofweek`, `month`, `is_weekend` | Menangkap pola jam, hari, dan bulan |
| Indikator aktivitas | `is_rush_morning`, `is_rush_evening`, `is_workhour` | Menandai periode aktivitas tinggi |
| Encoding siklik | `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos` | Membantu model memahami pola waktu yang berulang |
| Fitur musim | `season_simple`, `season_dry_flag` | Menangkap perbedaan musim kering dan basah |
| Fitur angin | `wind_u`, `wind_v` | Mengubah arah dan kecepatan angin menjadi komponen vektor |
| Fitur hujan | `has_rain` | Menandai ada atau tidaknya hujan |
| Lag PM2.5 | `pm25_lag_1`, `pm25_lag_6`, `pm25_lag_24` | Menangkap pengaruh PM2.5 masa lalu |
| Rolling PM2.5 | `pm25_roll_mean_*`, `pm25_roll_std_*` | Menangkap tren lokal dan variasi historis |
| Lag weather | `rain_lag_*`, `temperature_2m_lag_*`, `wind_speed_10m_lag_*` | Menangkap pengaruh cuaca masa lalu |
| Lag polutan pendukung | `pm10_work_lag_*`, `so2_work_lag_*`, dan lainnya | Menangkap hubungan antarpolutan |

Notebook ini membuat fitur secara terpisah untuk setiap horizon, karena kebutuhan fitur untuk prediksi 6 jam, 12 jam, dan 24 jam ke depan tidak selalu sama.

## Dataset Akhir per Horizon Prediksi

Notebook ini menghasilkan tiga dataset akhir berdasarkan horizon prediksi PM2.5:

| Horizon | Nama Dataset | Nama Target |
|---:|---|---|
| 6 jam | `dataset_h6.csv` | `target_pm25_t_plus_6` |
| 12 jam | `dataset_h12.csv` | `target_pm25_t_plus_12` |
| 24 jam | `dataset_h24.csv` | `target_pm25_t_plus_24` |

Setiap dataset berisi:

1. Kolom identitas waktu dan stasiun.
2. PM2.5 asli dan PM2.5 hasil imputasi.
3. Missing flag untuk menandai data yang awalnya kosong.
4. Fitur waktu dan musim.
5. Fitur cuaca.
6. Fitur lag.
7. Fitur rolling statistics.
8. Target masa depan sesuai horizon.

Dataset ini belum melakukan training model. Dataset ini hanya menyiapkan data agar siap digunakan pada notebook modeling berikutnya.

Catatan penting: karena data disusun sebagai time series, evaluasi model sebaiknya menggunakan split berbasis waktu, bukan random split.

## Membuka Dataset Master dan Membersihkan Struktur Awal

Cell ini membaca dataset utama:

```text
dataset_master_spku_weather.csv
```

Dataset ini merupakan hasil penggabungan antara data SPKU dan data weather pada notebook sebelumnya.

Tahapan yang dilakukan dalam cell ini adalah:

1. Mengimpor library utama, yaitu `numpy` dan `pandas`.
2. Membaca dataset menggunakan `pd.read_csv()`.
3. Mengubah kolom `datetime` dan `date` menjadi format datetime.
4. Mengurutkan data berdasarkan `station_slug` dan `datetime`.
5. Menentukan kolom target, kolom polutan, dan kolom weather.
6. Membuang kolom yang tidak digunakan.
7. Mengubah kolom numerik agar benar-benar terbaca sebagai angka.
8. Membatasi nilai PM2.5 maksimum sampai 500.

Kolom yang dibuang adalah:

| Kolom | Alasan Dibuang |
|---|---|
| `hc` | Pada EDA sebelumnya kolom ini tidak memiliki data yang berguna |
| `source_url` | Metadata scraping, tidak relevan untuk modeling |
| `source_file` | Metadata asal file, tidak relevan untuk modeling |
| `last_update` | Metadata update halaman, bukan fitur prediktif utama |
| `hour` | Redundan karena sudah ada `hour_num` atau akan dibuat ulang |

Output cell menunjukkan:

| Informasi | Nilai |
|---|---:|
| Nilai maksimum `pm25` setelah clipping | 500.0 |
| Shape awal setelah cleaning dasar | `(155880, 26)` |
| Missing value awal pada `pm25` | `75636` |

Artinya, setelah cleaning dasar, dataset masih memiliki **155.880 baris** dan **26 kolom**, tetapi kolom PM2.5 masih memiliki missing value yang cukup besar.

In [16]:
import numpy as np
import pandas as pd

# =========================
# LOAD
# =========================
DATA_PATH = "dataset_master_spku_weather.csv"
df = pd.read_csv(DATA_PATH)

# =========================
# BASIC CLEANING
# =========================
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.sort_values(["station_slug", "datetime"]).reset_index(drop=True)

TARGET_COL = "pm25"
POLLUTANT_COLS = ["pm10", "so2", "co", "o3", "no2"]
WEATHER_COLS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
]

DROP_COLS = ["hc", "source_url", "source_file", "last_update", "hour"]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=DROP_COLS)

for col in [TARGET_COL] + POLLUTANT_COLS + WEATHER_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["pm25"] = df["pm25"].clip(upper=500)

print("Nilai Maksimum pm25", df["pm25"].max())
print("Shape awal:", df.shape)
print("Missing pm25 awal:", df["pm25"].isna().sum())
df.head()

Nilai Maksimum pm25 500.0
Shape awal: (155880, 26)
Missing pm25 awal: 75636


,datetime,date,station_id,station_slug,station_name,lokasi,pm25,pm10,so2,co,...,hour_num,dayofweek,is_weekend,temperature_2m,relative_humidity_2m,precipitation,rain,surface_pressure,wind_speed_10m,wind_direction_10m
0,2022-10-01 00:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,NaN,NaN,NaN,NaN,...,0,5,1,26.1,82,0.0,0.0,1009.0,6.3,103
1,2022-10-01 01:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,NaN,NaN,NaN,NaN,...,1,5,1,25.8,84,0.0,0.0,1008.9,6.0,123
2,2022-10-01 02:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,NaN,NaN,NaN,NaN,...,2,5,1,25.2,87,0.0,0.0,1008.4,6.6,151
3,2022-10-01 03:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,NaN,NaN,NaN,NaN,...,3,5,1,24.8,90,0.0,0.0,1008.2,4.7,184
4,2022-10-01 04:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,NaN,NaN,NaN,NaN,...,4,5,1,24.5,92,0.0,0.0,1008.1,2.6,196


## Menangani Missing Value dengan Imputasi Bertingkat

Cell ini membuat beberapa fungsi bantu untuk menangani missing value pada weather, polutan pendukung, dan PM2.5.

### Fungsi Imputasi Weather

Fungsi `fill_weather_series()` digunakan untuk mengisi missing value pada variabel weather.

Strateginya adalah:

1. Interpolasi linear untuk missing value di bagian tengah.
2. Backward fill untuk mengisi missing value di awal.
3. Forward fill untuk mengisi missing value di akhir.

Weather dibuat kontinu karena variabel meteorologis biasanya berubah secara bertahap dari waktu ke waktu.

### Fungsi Imputasi Polutan Pendukung

Fungsi `fill_aux_series_reference_style()` digunakan untuk polutan pendukung seperti:

```text
pm10, so2, co, o3, no2
```

Strateginya adalah:

1. Forward fill terbatas maksimal 3 jam.
2. Interpolasi linear untuk missing value di bagian tengah.
3. Backward fill dan forward fill untuk bagian tepi.
4. Median stasiun jika masih ada missing value.

Hasil imputasi polutan pendukung disimpan dalam kolom baru dengan akhiran `_work`, misalnya:

| Kolom Asli | Kolom Kerja |
|---|---|
| `pm10` | `pm10_work` |
| `so2` | `so2_work` |
| `co` | `co_work` |
| `o3` | `o3_work` |
| `no2` | `no2_work` |

Kolom kerja ini digunakan untuk feature engineering, sedangkan kolom asli tetap dapat dipertahankan sebagai referensi.

### Fungsi Imputasi PM2.5

Fungsi `fill_pm25_full_reference_style()` digunakan untuk membuat PM2.5 versi penuh tanpa missing value.

Strateginya adalah:

1. Forward fill terbatas maksimal 3 jam.
2. Interpolasi linear untuk missing value internal.
3. Backward fill.
4. Forward fill.
5. Median stasiun jika masih ada missing value.

Sebelum imputasi dilakukan, nilai asli PM2.5 disimpan ke kolom:

```text
pm25_raw
```

Kemudian hasil imputasi digunakan untuk menggantikan kolom:

```text
pm25
```

Output cell menunjukkan:

| Kolom | Jumlah Missing |
|---|---:|
| `pm25_raw` | `75636` |
| `pm25` final | `0` |

Artinya, nilai PM2.5 asli masih menyimpan informasi missing value, sedangkan kolom `pm25` final sudah lengkap dan siap digunakan untuk membangun fitur time series serta target horizon.

Catatan penting: missing flag juga dibuat untuk target dan polutan pendukung. Kolom seperti `pm25_missing_flag` berguna agar model tetap bisa mengetahui apakah suatu nilai awalnya berasal dari observasi asli atau hasil imputasi.

In [17]:
# =========================
# HELPER FUNCTIONS
# =========================
def fill_weather_series(s):
    return s.interpolate(method="linear", limit_area="inside").bfill().ffill()

def fill_aux_series_reference_style(s):
    s = s.ffill(limit=3)
    s = s.interpolate(method="linear", limit_area="inside")
    s = s.bfill().ffill()

    if s.isna().any():
        med = s.median()
        if pd.notna(med):
            s = s.fillna(med)
    return s

def fill_pm25_full_reference_style(s):
    """
    PM2.5 full clean seperti notebook referensi:
    1) ffill terbatas
    2) interpolate internal
    3) bfill
    4) ffill
    5) median station
    """
    s = s.ffill(limit=3)
    s = s.interpolate(method="linear", limit_area="inside")
    s = s.bfill().ffill()

    if s.isna().any():
        med = s.median()
        if pd.notna(med):
            s = s.fillna(med)
    return s

# =========================
# BACKUP RAW PM2.5
# =========================
df["pm25_raw"] = df["pm25"]

# missing flags
for col in [TARGET_COL] + POLLUTANT_COLS:
    if col in df.columns:
        df[f"{col}_missing_flag"] = df[col].isna().astype("int8")

# =========================
# WEATHER IMPUTATION
# =========================
for col in WEATHER_COLS:
    if col in df.columns:
        df[col] = df.groupby("station_slug")[col].transform(fill_weather_series)

# =========================
# AUX POLLUTANT IMPUTATION
# =========================
for col in POLLUTANT_COLS:
    if col in df.columns:
        df[f"{col}_work"] = df.groupby("station_slug")[col].transform(fill_aux_series_reference_style)

        # fallback global median
        if df[f"{col}_work"].isna().any():
            global_med = df[f"{col}_work"].median()
            if pd.notna(global_med):
                df[f"{col}_work"] = df[f"{col}_work"].fillna(global_med)

# =========================
# PM2.5 FULL CLEAN
# =========================
df["pm25_clean_full"] = df.groupby("station_slug")["pm25"].transform(fill_pm25_full_reference_style)

# fallback global median kalau masih ada null
if df["pm25_clean_full"].isna().any():
    global_pm25_med = df["pm25_clean_full"].median()
    if pd.notna(global_pm25_med):
        df["pm25_clean_full"] = df["pm25_clean_full"].fillna(global_pm25_med)

# overwrite pm25 final seperti notebook referensi
df["pm25"] = df["pm25_clean_full"]
df.drop(columns=['pm25_clean_full'], axis=1, inplace=True)

print("Missing pm25_raw        :", df["pm25_raw"].isna().sum())
print("Missing pm25 final      :", df["pm25"].isna().sum())

Missing pm25_raw        : 75636
Missing pm25 final      : 0


## Membuat Fitur Waktu, Musim, Aktivitas, dan Angin

Cell ini membuat fitur tambahan yang berasal dari kolom waktu dan weather.

### Fitur Waktu Dasar

Beberapa fitur waktu dasar yang dibuat adalah:

| Fitur | Makna |
|---|---|
| `hour_num` | Jam dalam sehari, dari 0 sampai 23 |
| `dayofweek` | Hari dalam minggu, Senin = 0 |
| `month` | Bulan observasi |
| `day` | Tanggal dalam bulan |
| `is_weekend` | Penanda akhir pekan |

Fitur ini penting karena PM2.5 memiliki pola temporal, baik pola harian maupun pola musiman.

### Indikator Aktivitas

Notebook juga membuat fitur indikator aktivitas:

| Fitur | Makna |
|---|---|
| `is_rush_morning` | Bernilai 1 untuk jam 06.00 sampai 09.00 |
| `is_rush_evening` | Bernilai 1 untuk jam 16.00 sampai 19.00 |
| `is_workhour` | Bernilai 1 untuk jam 08.00 sampai 17.00 |

Fitur ini dibuat karena aktivitas manusia, lalu lintas, dan kegiatan perkotaan dapat berhubungan dengan perubahan konsentrasi PM2.5.

### Encoding Siklik

Notebook membuat encoding siklik untuk jam, hari, dan bulan:

| Fitur | Fungsi |
|---|---|
| `hour_sin`, `hour_cos` | Menangkap siklus 24 jam |
| `dow_sin`, `dow_cos` | Menangkap siklus mingguan |
| `month_sin`, `month_cos` | Menangkap siklus tahunan |

Encoding siklik lebih baik daripada angka biasa untuk data waktu berulang. Misalnya, jam 23 dan jam 0 sebenarnya berdekatan, tetapi jika hanya memakai angka biasa, model bisa menganggap keduanya sangat jauh.

### Fitur Musim

Notebook membagi musim secara sederhana:

| Musim | Bulan |
|---|---|
| `wet` | November, Desember, Januari, Februari, Maret, April |
| `dry` | Mei, Juni, Juli, Agustus, September, Oktober |

Hasilnya disimpan dalam:

| Kolom | Makna |
|---|---|
| `season_simple` | Label musim `wet` atau `dry` |
| `season_dry_flag` | Bernilai 1 jika musim kering |

### Fitur Angin dan Hujan

Arah angin dalam derajat diubah menjadi komponen vektor:

| Fitur | Makna |
|---|---|
| `wind_u` | Komponen angin horizontal berdasarkan arah angin |
| `wind_v` | Komponen angin vertikal berdasarkan arah angin |
| `has_rain` | Bernilai 1 jika `rain` lebih dari 0 |

Fitur `wind_u` dan `wind_v` dibuat karena arah angin dalam bentuk derajat bersifat melingkar. Dengan mengubahnya menjadi komponen vektor, model lebih mudah memahami arah dan kekuatan angin.

In [18]:
# =========================
# TEMPORAL FEATURES
# =========================
df["hour_num"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["is_weekend"] = (df["dayofweek"] >= 5).astype("int8")

df["is_rush_morning"] = df["hour_num"].between(6, 9).astype("int8")
df["is_rush_evening"] = df["hour_num"].between(16, 19).astype("int8")
df["is_workhour"] = df["hour_num"].between(8, 17).astype("int8")

df["hour_sin"] = np.sin(2 * np.pi * df["hour_num"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour_num"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["season_simple"] = np.where(df["month"].isin([11, 12, 1, 2, 3, 4]), "wet", "dry")
df["season_dry_flag"] = (df["season_simple"] == "dry").astype("int8")

# =========================
# WIND FEATURES
# =========================
wind_rad = np.deg2rad(df["wind_direction_10m"])
df["wind_u"] = df["wind_speed_10m"] * np.cos(wind_rad)
df["wind_v"] = df["wind_speed_10m"] * np.sin(wind_rad)
df["has_rain"] = (df["rain"] > 0).astype("int8")

df.head()

,datetime,date,station_id,station_slug,station_name,lokasi,pm25,pm10,so2,co,...,hour_cos,dow_sin,dow_cos,month_sin,month_cos,season_simple,season_dry_flag,wind_u,wind_v,has_rain
0,2022-10-01 00:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,77.0,NaN,NaN,NaN,...,1.000000,-0.974928,-0.222521,-0.866025,0.5,dry,1,-1.417192,6.138531,0
1,2022-10-01 01:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,77.0,NaN,NaN,NaN,...,0.965926,-0.974928,-0.222521,-0.866025,0.5,dry,1,-3.267834,5.032023,0
2,2022-10-01 02:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,77.0,NaN,NaN,NaN,...,0.866025,-0.974928,-0.222521,-0.866025,0.5,dry,1,-5.772490,3.199743,0
3,2022-10-01 03:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,77.0,NaN,NaN,NaN,...,0.707107,-0.974928,-0.222521,-0.866025,0.5,dry,1,-4.688551,-0.327855,0
4,2022-10-01 04:00:00,2022-10-01,4,dki1-bundaran-hi,DKI1 Bundaran HI,DKI Jakarta Air Quality Monitoring Station (SP...,77.0,NaN,NaN,NaN,...,0.500000,-0.974928,-0.222521,-0.866025,0.5,dry,1,-2.499280,-0.716657,0


## Merancang Fitur Berbasis Horizon Prediksi

Cell ini mendefinisikan konfigurasi fitur untuk tiga horizon prediksi, yaitu 6 jam, 12 jam, dan 24 jam ke depan.

Setiap horizon memiliki konfigurasi berbeda karena semakin jauh horizon prediksi, semakin panjang informasi historis yang dibutuhkan.

### Konfigurasi Horizon

| Horizon | Lag PM2.5 | Lag Weather | Lag Polutan | Rolling Window |
|---:|---|---|---|---|
| 6 jam | 1, 2, 3, 6, 12, 24 | 1, 3, 6, 12, 24 | 1, 3, 6, 12, 24 | 3, 6, 12, 24 |
| 12 jam | 1, 2, 3, 6, 12, 24, 48 | 1, 3, 6, 12, 24 | 1, 3, 6, 12, 24 | 3, 6, 12, 24 |
| 24 jam | 1, 3, 6, 12, 24, 48, 72 | 1, 3, 6, 12, 24, 48 | 1, 3, 6, 12, 24, 48 | 6, 12, 24, 48 |

### Target Horizon

Target dibuat dengan cara menggeser PM2.5 ke masa depan berdasarkan horizon:

| Horizon | Target |
|---:|---|
| 6 jam | `target_pm25_t_plus_6` |
| 12 jam | `target_pm25_t_plus_12` |
| 24 jam | `target_pm25_t_plus_24` |

Contohnya, untuk horizon 6 jam, target pada waktu `t` adalah nilai PM2.5 pada waktu `t+6`.

### Fitur Lag

Fitur lag dibuat dengan mengambil nilai masa lalu dari variabel tertentu. Contohnya:

| Fitur | Makna |
|---|---|
| `pm25_lag_1` | PM2.5 satu jam sebelumnya |
| `pm25_lag_6` | PM2.5 enam jam sebelumnya |
| `pm25_lag_24` | PM2.5 dua puluh empat jam sebelumnya |

Lag dibuat per stasiun menggunakan `groupby("station_slug")`, sehingga nilai dari satu stasiun tidak tercampur dengan stasiun lain.

### Fitur Rolling

Fitur rolling dibuat dari nilai masa lalu PM2.5 dan weather. Rolling menggunakan `shift(1)` sebelum menghitung statistik agar fitur hanya memakai informasi masa lalu, bukan nilai saat ini atau masa depan.

Fitur rolling PM2.5 yang dibuat meliputi:

| Fitur Rolling | Makna |
|---|---|
| `pm25_roll_mean_*` | Rata-rata PM2.5 pada window tertentu |
| `pm25_roll_std_*` | Standar deviasi PM2.5 pada window tertentu |
| `pm25_roll_min_*` | Nilai minimum PM2.5 pada window tertentu |
| `pm25_roll_max_*` | Nilai maksimum PM2.5 pada window tertentu |

Untuk weather, rolling yang dibuat meliputi rolling sum hujan dan rolling mean untuk suhu, kelembapan, serta kecepatan angin.

### Fitur Profil Stasiun

Cell ini juga membuat fitur:

| Fitur | Makna |
|---|---|
| `station_hour_mean_pm25` | Rata-rata PM2.5 per stasiun dan jam |
| `station_month_mean_pm25` | Rata-rata PM2.5 per stasiun dan bulan |

Catatan metodologis: fitur profil ini dihitung dari seluruh dataset. Jika digunakan pada evaluasi berbasis waktu, fitur seperti ini berpotensi membawa informasi dari masa depan. Untuk eksperimen final, fitur profil sebaiknya dihitung hanya dari data training pada setiap fold agar tidak terjadi data leakage.

### Final Cleanup

Setelah semua fitur dibuat, missing value pada fitur numerik diisi menggunakan median per stasiun, lalu median global, dan terakhir 0 jika masih kosong.

Baris yang tidak memiliki target masa depan dibuang karena tidak bisa digunakan untuk supervised learning.

In [19]:
HORIZON_CONFIG = {
    6: {
        "pm25_lags": [1, 2, 3, 6, 12, 24],
        "weather_lags": [1, 3, 6, 12, 24],
        "pollutant_lags": [1, 3, 6, 12, 24],
        "roll_windows": [3, 6, 12, 24],
    },
    12: {
        "pm25_lags": [1, 2, 3, 6, 12, 24, 48],
        "weather_lags": [1, 3, 6, 12, 24],
        "pollutant_lags": [1, 3, 6, 12, 24],
        "roll_windows": [3, 6, 12, 24],
    },
    24: {
        "pm25_lags": [1, 3, 6, 12, 24, 48, 72],
        "weather_lags": [1, 3, 6, 12, 24, 48],
        "pollutant_lags": [1, 3, 6, 12, 24, 48],
        "roll_windows": [6, 12, 24, 48],
    },
}

WEATHER_LAG_COLS = [
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "surface_pressure",
    "wind_speed_10m",
    "wind_u",
    "wind_v",
]

POLLUTANT_WORK_COLS = [f"{c}_work" for c in POLLUTANT_COLS if f"{c}_work" in df.columns]

def fill_engineered_numeric(frame, by="station_slug"):
    out = frame.copy()
    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if out[col].isna().any():
            station_med = out.groupby(by)[col].transform("median")
            out[col] = out[col].fillna(station_med)

            if out[col].isna().any():
                global_med = out[col].median()
                if pd.notna(global_med):
                    out[col] = out[col].fillna(global_med)
                else:
                    out[col] = out[col].fillna(0)
    return out

def make_horizon_dataset(base_df, horizon):
    cfg = HORIZON_CONFIG[horizon]
    out = base_df.copy()
    g = out.groupby("station_slug")

    # target dari PM2.5 full clean
    target_col = f"target_pm25_t_plus_{horizon}"
    out[target_col] = g["pm25"].shift(-horizon)

    # pm25 lags dari PM2.5 full clean
    for lag in cfg["pm25_lags"]:
        out[f"pm25_lag_{lag}"] = g["pm25"].shift(lag)

    if 1 in cfg["pm25_lags"] and 2 in cfg["pm25_lags"]:
        out["pm25_diff_1"] = out["pm25_lag_1"] - out["pm25_lag_2"]
    if 24 in cfg["pm25_lags"] and 1 in cfg["pm25_lags"]:
        out["pm25_diff_24"] = out["pm25_lag_1"] - out["pm25_lag_24"]

    # rolling dari PM2.5 full clean, tetap past-only
    for w in cfg["roll_windows"]:
        out[f"pm25_roll_mean_{w}"] = g["pm25"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).mean()
        )
        out[f"pm25_roll_std_{w}"] = g["pm25"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).std()
        )
        out[f"pm25_roll_min_{w}"] = g["pm25"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).min()
        )
        out[f"pm25_roll_max_{w}"] = g["pm25"].transform(
            lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).max()
        )

    # weather lags
    for col in WEATHER_LAG_COLS:
        if col in out.columns:
            for lag in cfg["weather_lags"]:
                out[f"{col}_lag_{lag}"] = g[col].shift(lag)

    # weather rolling
    for w in cfg["roll_windows"]:
        if "rain" in out.columns:
            out[f"rain_roll_sum_{w}"] = g["rain"].transform(
                lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).sum()
            )
        if "temperature_2m" in out.columns:
            out[f"temperature_2m_roll_mean_{w}"] = g["temperature_2m"].transform(
                lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).mean()
            )
        if "relative_humidity_2m" in out.columns:
            out[f"relative_humidity_2m_roll_mean_{w}"] = g["relative_humidity_2m"].transform(
                lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).mean()
            )
        if "wind_speed_10m" in out.columns:
            out[f"wind_speed_10m_roll_mean_{w}"] = g["wind_speed_10m"].transform(
                lambda s: s.shift(1).rolling(w, min_periods=max(1, w // 2)).mean()
            )

    # auxiliary pollutant lags
    for col in POLLUTANT_WORK_COLS:
        for lag in cfg["pollutant_lags"]:
            out[f"{col}_lag_{lag}"] = g[col].shift(lag)

    # station history profile
    out["station_hour_mean_pm25"] = out.groupby(["station_slug", "hour_num"])["pm25"].transform("mean")
    out["station_month_mean_pm25"] = out.groupby(["station_slug", "month"])["pm25"].transform("mean")

    # final cleanup engineered numeric
    protected = {"pm25_raw", target_col}
    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    feature_fill_cols = [c for c in numeric_cols if c not in protected]

    temp = out[["station_slug"] + feature_fill_cols].copy()
    temp = fill_engineered_numeric(temp, by="station_slug")

    for col in feature_fill_cols:
        out[col] = temp[col]

    # drop row tanpa target
    out = out.dropna(subset=[target_col]).reset_index(drop=True)

    return out, target_col

## Membentuk Dataset Final untuk Tiga Horizon

Cell ini menjalankan fungsi `make_horizon_dataset()` untuk tiga horizon prediksi:

1. Horizon 6 jam.
2. Horizon 12 jam.
3. Horizon 24 jam.

Kode menghasilkan tiga dataframe:

| Dataset | Target |
|---|---|
| `dataset_h6` | `target_pm25_t_plus_6` |
| `dataset_h12` | `target_pm25_t_plus_12` |
| `dataset_h24` | `target_pm25_t_plus_24` |

Output cell menunjukkan:

| Dataset | Shape | Target |
|---|---:|---|
| `dataset_h6` | `(155850, 155)` | `target_pm25_t_plus_6` |
| `dataset_h12` | `(155820, 156)` | `target_pm25_t_plus_12` |
| `dataset_h24` | `(155760, 167)` | `target_pm25_t_plus_24` |

Jumlah baris semakin sedikit ketika horizon semakin panjang. Hal ini wajar karena semakin jauh horizon prediksi, semakin banyak baris terakhir pada setiap stasiun yang tidak memiliki target masa depan.

Contohnya:

| Horizon | Dampak pada Baris Akhir |
|---:|---|
| 6 jam | 6 jam terakhir per stasiun tidak punya target |
| 12 jam | 12 jam terakhir per stasiun tidak punya target |
| 24 jam | 24 jam terakhir per stasiun tidak punya target |

Cell ini juga mengecek missing value pada kolom `pm25` di masing-masing dataset. Output menunjukkan bahwa `pm25` sudah tidak memiliki missing value pada semua dataset.

Pada output juga muncul `PerformanceWarning` dari pandas karena banyak kolom ditambahkan satu per satu. Warning ini tidak menghentikan proses dan dataset tetap berhasil dibuat. Namun, jika notebook ingin dioptimasi, pembuatan fitur bisa diperbaiki dengan mengumpulkan fitur dalam list dataframe lalu digabung menggunakan `pd.concat(axis=1)`.

In [20]:
dataset_h6, target_h6 = make_horizon_dataset(df, 6)
dataset_h12, target_h12 = make_horizon_dataset(df, 12)
dataset_h24, target_h24 = make_horizon_dataset(df, 24)

print("dataset_h6 :", dataset_h6.shape, "| target:", target_h6)
print("dataset_h12:", dataset_h12.shape, "| target:", target_h12)
print("dataset_h24:", dataset_h24.shape, "| target:", target_h24)

print("\nMissing pm25 di masing-masing dataset:")
print("h6 :", dataset_h6["pm25"].isna().sum())
print("h12:", dataset_h12["pm25"].isna().sum())
print("h24:", dataset_h24["pm25"].isna().sum())

/tmp/ipykernel_1101/3894189974.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{col}_lag_{lag}"] = g[col].shift(lag)
/tmp/ipykernel_1101/3894189974.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{col}_lag_{lag}"] = g[col].shift(lag)
/tmp/ipykernel_1101/3894189974.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

dataset_h6 : (155850, 155) | target: target_pm25_t_plus_6
dataset_h12: (155820, 156) | target: target_pm25_t_plus_12
dataset_h24: (155760, 167) | target: target_pm25_t_plus_24

Missing pm25 di masing-masing dataset:
h6 : 0
h12: 0
h24: 0


## Memeriksa Struktur Dataset Horizon 6 Jam

Cell ini mengecek bentuk dan daftar kolom pada `dataset_h6`.

Output menunjukkan:

```text
(155850, 155)
```

Artinya, dataset horizon 6 jam memiliki **155.850 baris** dan **155 kolom**.

Dataset ini digunakan untuk memprediksi PM2.5 **6 jam ke depan** dengan target:

```text
target_pm25_t_plus_6
```

Pemeriksaan daftar kolom penting untuk memastikan bahwa semua fitur yang dibutuhkan sudah berhasil dibuat, seperti:

1. Fitur identitas waktu dan stasiun.
2. Fitur weather.
3. Fitur musim dan aktivitas.
4. Fitur lag PM2.5.
5. Fitur rolling PM2.5.
6. Fitur lag weather.
7. Fitur lag polutan pendukung.
8. Kolom target horizon 6 jam.

Karena horizon 6 jam termasuk horizon pendek, fitur lag jangka pendek seperti `pm25_lag_1`, `pm25_lag_2`, `pm25_lag_3`, dan `pm25_lag_6` menjadi sangat penting.

In [21]:
print(dataset_h6.shape)
dataset_h6.columns.tolist()

(155850, 155)


['datetime',
 'date',
 'station_id',
 'station_slug',
 'station_name',
 'lokasi',
 'pm25',
 'pm10',
 'so2',
 'co',
 'o3',
 'no2',
 'kategori',
 'year',
 'month',
 'day',
 'hour_num',
 'dayofweek',
 'is_weekend',
 'temperature_2m',
 'relative_humidity_2m',
 'precipitation',
 'rain',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m',
 'pm25_raw',
 'pm25_missing_flag',
 'pm10_missing_flag',
 'so2_missing_flag',
 'co_missing_flag',
 'o3_missing_flag',
 'no2_missing_flag',
 'pm10_work',
 'so2_work',
 'co_work',
 'o3_work',
 'no2_work',
 'is_rush_morning',
 'is_rush_evening',
 'is_workhour',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'season_simple',
 'season_dry_flag',
 'wind_u',
 'wind_v',
 'has_rain',
 'target_pm25_t_plus_6',
 'pm25_lag_1',
 'pm25_lag_2',
 'pm25_lag_3',
 'pm25_lag_6',
 'pm25_lag_12',
 'pm25_lag_24',
 'pm25_diff_1',
 'pm25_diff_24',
 'pm25_roll_mean_3',
 'pm25_roll_std_3',
 'pm25_roll_min_3',
 'pm25_roll_max_3',
 'pm25_roll_m

## Memeriksa Struktur Dataset Horizon 12 Jam

Cell ini mengecek bentuk dan daftar kolom pada `dataset_h12`.

Output menunjukkan:

```text
(155820, 156)
```

Artinya, dataset horizon 12 jam memiliki **155.820 baris** dan **156 kolom**.

Dataset ini digunakan untuk memprediksi PM2.5 **12 jam ke depan** dengan target:

```text
target_pm25_t_plus_12
```

Jumlah kolom pada dataset H12 sedikit lebih banyak dibanding H6 karena konfigurasi H12 menggunakan tambahan lag PM2.5 yang lebih panjang, yaitu `pm25_lag_48`.

Horizon 12 jam dapat dianggap sebagai horizon menengah. Pada horizon ini, model tidak hanya membutuhkan informasi PM2.5 beberapa jam terakhir, tetapi juga pola setengah hari hingga dua hari sebelumnya.

In [22]:
print(dataset_h12.shape)
dataset_h12.columns.tolist()

(155820, 156)


['datetime',
 'date',
 'station_id',
 'station_slug',
 'station_name',
 'lokasi',
 'pm25',
 'pm10',
 'so2',
 'co',
 'o3',
 'no2',
 'kategori',
 'year',
 'month',
 'day',
 'hour_num',
 'dayofweek',
 'is_weekend',
 'temperature_2m',
 'relative_humidity_2m',
 'precipitation',
 'rain',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m',
 'pm25_raw',
 'pm25_missing_flag',
 'pm10_missing_flag',
 'so2_missing_flag',
 'co_missing_flag',
 'o3_missing_flag',
 'no2_missing_flag',
 'pm10_work',
 'so2_work',
 'co_work',
 'o3_work',
 'no2_work',
 'is_rush_morning',
 'is_rush_evening',
 'is_workhour',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'season_simple',
 'season_dry_flag',
 'wind_u',
 'wind_v',
 'has_rain',
 'target_pm25_t_plus_12',
 'pm25_lag_1',
 'pm25_lag_2',
 'pm25_lag_3',
 'pm25_lag_6',
 'pm25_lag_12',
 'pm25_lag_24',
 'pm25_lag_48',
 'pm25_diff_1',
 'pm25_diff_24',
 'pm25_roll_mean_3',
 'pm25_roll_std_3',
 'pm25_roll_min_3',
 'pm25_roll_max_

## Memeriksa Struktur Dataset Horizon 24 Jam

Cell ini mengecek bentuk dan daftar kolom pada `dataset_h24`.

Output menunjukkan:

```text
(155760, 167)
```

Artinya, dataset horizon 24 jam memiliki **155.760 baris** dan **167 kolom**.

Dataset ini digunakan untuk memprediksi PM2.5 **24 jam ke depan** dengan target:

```text
target_pm25_t_plus_24
```

Dataset H24 memiliki jumlah kolom paling banyak karena menggunakan lag dan rolling window yang lebih panjang, seperti:

| Kelompok | Contoh |
|---|---|
| Lag PM2.5 panjang | `pm25_lag_48`, `pm25_lag_72` |
| Lag weather panjang | weather lag sampai 48 jam |
| Lag polutan panjang | pollutant lag sampai 48 jam |
| Rolling window panjang | rolling 6, 12, 24, dan 48 jam |

Horizon 24 jam lebih sulit dibanding H6 dan H12 karena model harus memprediksi satu hari ke depan. Oleh karena itu, fitur musiman, fitur jam, fitur rolling, dan fitur lag panjang menjadi semakin penting.

In [23]:
print(dataset_h24.shape)
dataset_h24.columns.tolist()

(155760, 167)


['datetime',
 'date',
 'station_id',
 'station_slug',
 'station_name',
 'lokasi',
 'pm25',
 'pm10',
 'so2',
 'co',
 'o3',
 'no2',
 'kategori',
 'year',
 'month',
 'day',
 'hour_num',
 'dayofweek',
 'is_weekend',
 'temperature_2m',
 'relative_humidity_2m',
 'precipitation',
 'rain',
 'surface_pressure',
 'wind_speed_10m',
 'wind_direction_10m',
 'pm25_raw',
 'pm25_missing_flag',
 'pm10_missing_flag',
 'so2_missing_flag',
 'co_missing_flag',
 'o3_missing_flag',
 'no2_missing_flag',
 'pm10_work',
 'so2_work',
 'co_work',
 'o3_work',
 'no2_work',
 'is_rush_morning',
 'is_rush_evening',
 'is_workhour',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'season_simple',
 'season_dry_flag',
 'wind_u',
 'wind_v',
 'has_rain',
 'target_pm25_t_plus_24',
 'pm25_lag_1',
 'pm25_lag_3',
 'pm25_lag_6',
 'pm25_lag_12',
 'pm25_lag_24',
 'pm25_lag_48',
 'pm25_lag_72',
 'pm25_diff_24',
 'pm25_roll_mean_6',
 'pm25_roll_std_6',
 'pm25_roll_min_6',
 'pm25_roll_max_6',
 'pm25_roll

In [24]:
print(dataset_h6[["pm25_raw" , "pm25"]].isna().sum())
print(dataset_h12[["pm25_raw", "pm25"]].isna().sum())
print(dataset_h24[["pm25_raw", "pm25"]].isna().sum())

pm25_raw    75631
pm25            0
dtype: int64
pm25_raw    75626
pm25            0
dtype: int64
pm25_raw    75626
pm25            0
dtype: int64


## Menyimpan Dataset Modeling ke File CSV

Cell ini menyimpan tiga dataset final ke dalam file CSV.

File yang disimpan adalah:

| Dataset | File Output |
|---|---|
| `dataset_h6` | `dataset_h6.csv` |
| `dataset_h12` | `dataset_h12.csv` |
| `dataset_h24` | `dataset_h24.csv` |

Output cell menunjukkan:

```text
Saved:
- dataset_h6.csv
- dataset_h12.csv
- dataset_h24.csv
```

Artinya, semua dataset horizon berhasil disimpan dan siap digunakan pada notebook berikutnya untuk proses training dan evaluasi model.

File ini dapat langsung dipakai untuk eksperimen seperti:

1. Baseline LightGBM per horizon.
2. Perbandingan performa H6, H12, dan H24.
3. Evaluasi feature importance.
4. Ablation study fitur weather, polutan pendukung, dan fitur waktu.
5. Ensemble atau stacking model.

In [25]:
dataset_h6.to_csv("dataset_h6.csv", index=False)
dataset_h12.to_csv("dataset_h12.csv", index=False)
dataset_h24.to_csv("dataset_h24.csv", index=False)

print("Saved:")
print("- dataset_h6.csv")
print("- dataset_h12.csv")
print("- dataset_h24.csv")

Saved:
- dataset_h6.csv
- dataset_h12.csv
- dataset_h24.csv


# Struktur File Output Notebook

Notebook ini tidak membuat folder output khusus. Semua file hasil preprocessing modeling disimpan langsung di working directory.

Input utama notebook ini adalah:

```text
dataset_master_spku_weather.csv
```

Output utama notebook ini adalah:

```text
dataset_h6.csv
dataset_h12.csv
dataset_h24.csv
```

Struktur file dapat diringkas sebagai berikut:

```text
project/
│
├── dataset_master_spku_weather.csv
│
├── dataset_h6.csv
├── dataset_h12.csv
└── dataset_h24.csv
```

Penjelasan file:

| File | Isi | Kegunaan |
|---|---|---|
| `dataset_master_spku_weather.csv` | Dataset master hasil merge SPKU dan weather | Input utama preprocessing |
| `dataset_h6.csv` | Dataset modeling untuk target PM2.5 6 jam ke depan | Input model horizon 6 jam |
| `dataset_h12.csv` | Dataset modeling untuk target PM2.5 12 jam ke depan | Input model horizon 12 jam |
| `dataset_h24.csv` | Dataset modeling untuk target PM2.5 24 jam ke depan | Input model horizon 24 jam |

Ringkasan ukuran dataset akhir:

| Dataset | Shape | Target |
|---|---:|---|
| `dataset_h6.csv` | `(155850, 155)` | `target_pm25_t_plus_6` |
| `dataset_h12.csv` | `(155820, 156)` | `target_pm25_t_plus_12` |
| `dataset_h24.csv` | `(155760, 167)` | `target_pm25_t_plus_24` |

Secara keseluruhan, notebook ini menghasilkan dataset final yang sudah siap digunakan untuk training model forecasting PM2.5. Tahap berikutnya adalah membuat pipeline modeling, menentukan fitur yang digunakan, melakukan split berbasis waktu, melatih model, dan mengevaluasi performa setiap horizon.